In [ ]:
# import toml
# from pathlib import Path
# config_path = Path("../../../../configuration")
# input_config = toml.load(config_path/ "input_configuration.toml")
# summary_config = toml.load(config_path/ "configuration/summary_configuration.toml")

In [ ]:
import numpy as np
import plotly.express as px
import pandas as pd
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

In [ ]:
hh = util.get_hh_data(summary_config).to_pandas()
person = util.get_person_data(summary_config).to_pandas()
trip = util.get_trip_data(summary_config).to_pandas()

In [3]:
df_trip = trip.copy()
df_hh = hh.copy()
df_person = person.copy()

df_person = df_person.merge(df_hh,
                          how='left', on=['hhno','source']) # get auto ownership from hh data

df_trip = df_trip.merge(df_person, how='left', on=['pno','hhno','source'])

In [4]:
es_trip = df_trip.loc[(df_trip['dpurp']==3)].copy()

In [5]:
df_plot = es_trip.groupby(['source','mode_label'])['trexpfac'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source'], group_keys=False)['trexpfac']. \
    apply(lambda x: x / float(x.sum()))

df_plot_ct = es_trip.groupby(['source','mode_label'])['trexpfac'].count().reset_index(). \
    rename(columns={'trexpfac':'sample count'})
df_plot = df_plot.merge(df_plot_ct, on=['source','mode_label'])

fig = px.bar(df_plot.sort_values(by=['source']), x="mode_label", y="percentage", color="source",
             barmode="group",hover_data=['sample count'],title="escort trip mode")
fig.update_layout(height=400, width=700, font=dict(size=11),
                  xaxis = dict(dtick = 1, categoryorder='category ascending'),
                  yaxis=dict(tickformat=".2%"))
fig.show()

### mode choice by segment

In [6]:
def plot_mode_choice(df: pd.DataFrame, grp_var: str, order_list: dict, title_name: str, n_nol: int, height=400, width=800):
    df_plot = df.groupby(['source',grp_var,'mode_label'])['trexpfac'].sum().reset_index()
    df_plot['percentage'] = df_plot.groupby(['source',grp_var], group_keys=False)['trexpfac']. \
        apply(lambda x: x / float(x.sum()))

    df_plot_ct = df.groupby(['source',grp_var,'mode_label'])['trexpfac'].count().reset_index(). \
        rename(columns={'trexpfac':'sample count'})
    df_plot = df_plot.merge(df_plot_ct, on=['source',grp_var,'mode_label'])

    fig = px.bar(df_plot.sort_values(['source','mode_label']),
                 x="percentage", y="mode_label", color="source",barmode="group",
                 facet_col=grp_var, facet_col_wrap=n_nol, orientation='h',
                 hover_data=['sample count'],
                 category_orders=order_list,
                 title="escort trip mode choice by " + title_name)
    fig.update_layout(height=height, width=width)
    fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
    fig.for_each_xaxis(lambda a: a.update(tickformat = ".1%"))
    fig.show()

In [8]:
plot_mode_choice(es_trip,"pptyp_label",
                 {"pptyp_label":util.pptyp_cat.values(),
                  "mode_label":util.mode_cat.values()},
                 "person type",
                 2,1000)

In [9]:
plot_mode_choice(es_trip,"hhsize_4+",
                 {"hhsize_4+":["1","2","3","4+"],
                  "mode_label":util.mode_cat.values()},
                 "household size",2,600)

In [10]:
plot_mode_choice(es_trip.loc[es_trip['hhvehs_4+']!="-1"],
                 "hhvehs_4+",
                 {"hhvehs_4+":["0","1","2","3","4+"],
                  "mode_label":util.mode_cat.values()},
                 "auto ownership",2,800)

In [11]:
plot_mode_choice(es_trip,
                 "hhcu5",
                 {"mode_label":util.mode_cat.values()},
                 "HH with Children Under 5",2,800)

In [12]:
plot_mode_choice(es_trip,
                 "hh515",
                 {"mode_label":util.mode_cat.values()},
                 "HH with Children 5-15",2,800)

In [13]:
plot_mode_choice(es_trip,
                 "hhhsc",
                 {"mode_label":util.mode_cat.values()},
                 "HH with Driving Age Students",2,800)

In [15]:
plot_mode_choice(es_trip.loc[es_trip['auto_available_drivers'] != "no dirver"], "auto_available_drivers",
                 {"auto_available_drivers": ["no car", "cars fewer than drivers", "enough cars"],
                  "mode_label":util.mode_cat.values()},
                 "auto availability (driver, showing only households with at least one driver)", 3, 500, 1000)